# Numerical derivatives

Finite difference schemes approximate derivatives by sampling the function at nearby points. This notebook walks through the forward, backward, and central difference rules, compares their behavior on familiar functions, and demonstrates how errors change with the step size `h`.

### Difference schemes and error scaling

- **Forward difference** uses `f(x+h)` and `f(x)`; the truncation error is \(O(h)\).
- **Backward difference** uses `f(x)` and `f(x-h)`; it is also \(O(h)\).
- **Central difference** uses both `f(x+h)` and `f(x-h)`; the leading error term cancels, so it is \(O(h^2)\) when the function is smooth.

Because central differencing has higher-order truncation error, it usually gives a better approximation for moderate `h`. Later we will see that making `h` too small invites rounding error, so there is a sweet spot in practice.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

assets_dir = root / "assets" / "figures"
assets_dir.mkdir(parents=True, exist_ok=True)

### Sin and cos on a shared grid

We plot `sin(x)` alongside the true cosine and a central difference approximation of the derivative. This illustrates how the approximated derivative tracks the analytic derivative across the domain.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from calculus_with_python.examples import sin, cos
from calculus_with_python.grids import make_grid
from calculus_with_python.differentiation import difference_quotient

x = make_grid(-2.0 * np.pi, 2.0 * np.pi, 900)
sin_vals = sin(x)
cos_true = cos(x)
cos_central = difference_quotient(sin, x, method="central", h=1e-5)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, sin_vals, label="sin(x)")
ax.plot(x, cos_true, label="cos(x) (true)", linestyle="--", alpha=0.8)
ax.plot(x, cos_central, label="central difference", linewidth=2, alpha=0.7)
ax.set_title("Derivative of sin(x)")
ax.set_xlabel("x")
ax.set_ylabel("value")
ax.legend()
ax.grid(True)
fig.savefig(assets_dir / "numerical_derivatives_sin_cos.png", dpi=150, bbox_inches="tight")
plt.close(fig)
fig

### A quadratic example

Quadratic polynomials have a linear derivative, so `f(x) = x^2` differentiates to `f'(x) = 2x`. We plot both the analytic derivative and the central difference approximation to visualize the discrepancy on a fixed grid.

In [ ]:
from calculus_with_python.examples import poly

quadratic = poly(0.0, 0.0, 1.0)  # x^2
x = make_grid(-3.0, 3.0, 600)
true_derivative = 2.0 * x
approx_derivative = difference_quotient(quadratic, x, method="central", h=1e-5)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, true_derivative, label="2x (analytic)", linewidth=2)
ax.plot(x, approx_derivative, label="central difference", linestyle="--")
ax.set_title("Derivative of x^2")
ax.set_xlabel("x")
ax.set_ylabel("f'(x)")
ax.legend()
ax.grid(True)
fig.savefig(assets_dir / "numerical_derivatives_quadratic.png", dpi=150, bbox_inches="tight")
plt.close(fig)
fig

### Error vs step size `h`

The truncation error for a central difference scales like \(O(h^2)\), so the approximation improves as we shrink `h`. However, rounding error grows when `h` becomes comparable to the machine epsilon because subtraction loses precision. Plotting the absolute error between the central difference of `sin(x)` at a fixed point and the analytic cosine shows this tradeoff.

In [ ]:
x0 = 1.0
h_values = np.logspace(-15, -1, 80)
errors = []
for h in h_values:
    approx_derivative = difference_quotient(sin, x0, h=h, method="central")
    errors.append(abs(approx_derivative - np.cos(x0)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(h_values, errors, marker="o", markersize=4)
ax.set_title("Central difference error vs step size")
ax.set_xlabel("h")
ax.set_ylabel("|approx - cos(x0)|")
ax.grid(True, which="both", linestyle=":")
fig.savefig(assets_dir / "numerical_derivatives_error_vs_h.png", dpi=150, bbox_inches="tight")
plt.close(fig)
fig

At very small steps `h`, rounding error dominates because subtraction between nearly equal numbers loses digits. You can see the error curve bottom out and then turn upward once rounding noise overpowers the `O(h^2)` truncation error. Choosing `h` in the sweet spot between both regions yields the most accurate estimate.

### Symbolic derivative as ground truth

When SymPy is available, treat the symbolic derivative as the ground truth and compare it directly against the finite-difference estimate. The code below lambdifies both expressions, plots the curves, and shows the absolute error.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

try:
    from calculus_with_python.symbolic import lambdify_expr, symbolic_derivative, to_symbol
except ImportError:
    print('Install the optional symbolic extra (poetry install -E symbolic) to run this section.')
else:
    from calculus_with_python.differentiation import difference_quotient
    from calculus_with_python.grids import make_grid

    expr = 'sin(x) + x**2'
    sym_expr = to_symbol(expr)
    sym_deriv = symbolic_derivative(expr, order=1)
    sym_fn = lambdify_expr(sym_deriv)
    base_fn = lambdify_expr(sym_expr)

    x_sym = make_grid(-2.0, 2.0, 500)
    num_deriv = difference_quotient(base_fn, x_sym, h=1e-5, method='central')
    sym_values = np.asarray(sym_fn(x_sym), dtype=np.float64)

    print('Symbolic expression:', sym_expr)
    print('Symbolic derivative:', sym_deriv)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    ax1.plot(x_sym, sym_values, label='symbolic', linewidth=2)
    ax1.plot(x_sym, num_deriv, label='numerical (central)', linestyle='--', linewidth=1.5)
    ax1.set_title('Symbolic vs. numerical derivative for ' + expr)
    ax1.set_ylabel('f"(x)')
    ax1.legend()
    ax1.grid(True)

    error = np.abs(sym_values - num_deriv)
    ax2.plot(x_sym, error, color='tab:orange')
    ax2.set_ylabel('|error|')
    ax2.set_xlabel('x')
    ax2.set_title('Absolute error between symbolic and numerical derivatives')
    ax2.grid(True)

    fig.tight_layout()
    fig.savefig(assets_dir / 'symbolic_derivative_comparison.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    fig

### Wrap-up

The symbolic helpers provide a high-fidelity ground truth for finite differences and highlight where the numerical derivative is most challenged. When SymPy is missing, the rest of the lesson still runs and this section prints installation instructions.